In [1]:
# fetch BRO borehole data via API
import requests
import pandas as pd

# # BRO REST API for ground temperature / borehole data
# url = "https://publiek.broservices.nl/sr/gmw/v2/objects"
# params = {
#     "bbox": "4.5,52.0,5.5,53.0",  # bounding box for e.g. Utrecht region
#     "observationType": "groundTemperature"
# }
# response = requests.get(url, params=params)
# data = response.json()

In [2]:
import requests
import geopandas as gpd
from io import BytesIO

url = "https://service.pdok.nl/cbs/gebiedsindelingen/2024/wfs/v1_0"

params = {
    "service":      "WFS",
    "version":      "2.0.0",
    "request":      "GetFeature",
    "typeName":     "gebiedsindelingen:gemeente_gegeneraliseerd",
    "outputFormat": "application/json",
    "count":        1
}

response = requests.get(url, params=params)
print("Status:", response.status_code)

gdf = gpd.read_file(BytesIO(response.content))
print("Columns:", gdf.columns.tolist())
print("\nSample row:")
print(gdf.iloc[0].to_dict())

Status: 200
Columns: ['statcode', 'jrstatcode', 'statnaam', 'rubriek', 'id', 'geometry']

Sample row:
{'statcode': 'GM0014', 'jrstatcode': '2024GM0014', 'statnaam': 'Groningen', 'rubriek': 'gemeente', 'id': 1, 'geometry': <MULTIPOLYGON (((245194.691 592594.007, 245344.71 592467.73, 245358.268 5925...>}


In [3]:
import requests
import geopandas as gpd
from io import BytesIO

url = "https://service.pdok.nl/cbs/gebiedsindelingen/2024/wfs/v1_0"

def get_gemeente_by_code(gemeente_code):
    """
    Fetch municipality boundary using CBS gemeente code e.g. 'GM0344'
    """
    params = {
        "service":      "WFS",
        "version":      "2.0.0",
        "request":      "GetFeature",
        "typeName":     "gebiedsindelingen:gemeente_gegeneraliseerd",
        "CQL_FILTER":   f"statcode='{gemeente_code}'",  # full GM0344 format, no stripping needed
        "outputFormat": "application/json"
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        print(f"Error {response.status_code}: {response.text[:200]}")
        return None

    gdf = gpd.read_file(BytesIO(response.content))

    if gdf.empty:
        print(f"No results for {gemeente_code}")
        return None

    print(f"Found:  {gdf['statnaam'].values[0]}")
    print(f"Code:   {gdf['statcode'].values[0]}")
    print(f"Bounds: {gdf.total_bounds}")
    return gdf

# Test it
utrecht = get_gemeente_by_code("GM0344")

Found:  Groningen
Code:   GM0014
Bounds: [ 13565.39999999 306846.198      278026.09       619231.64870001]


In [4]:
# First check: does the filter get applied at all?
params = {
    "service":      "WFS",
    "version":      "2.0.0",
    "request":      "GetFeature",
    "typeName":     "gebiedsindelingen:gemeente_gegeneraliseerd",
    "CQL_FILTER":   "statcode='GM0344'",
    "outputFormat": "application/json",
    "count":        5
}

response = requests.get(url, params=params)
gdf = gpd.read_file(BytesIO(response.content))
print(f"Records returned: {len(gdf)}")
print(gdf[["statcode", "statnaam"]])

Records returned: 5
  statcode     statnaam
0   GM0014    Groningen
1   GM0034       Almere
2   GM0037  Stadskanaal
3   GM0047      Veendam
4   GM0050     Zeewolde


In [5]:
def get_all_gemeenten(page_size=100):
    """Download all gemeenten with pagination, filter locally"""
    all_gdfs = []
    start_index = 0

    while True:
        params = {
            "service":      "WFS",
            "version":      "2.0.0",
            "request":      "GetFeature",
            "typeName":     "gebiedsindelingen:gemeente_gegeneraliseerd",
            "outputFormat": "application/json",
            "count":        page_size,
            "startIndex":   start_index
        }

        response = requests.get(url, params=params)
        gdf = gpd.read_file(BytesIO(response.content))

        if gdf.empty:
            break

        all_gdfs.append(gdf)
        print(f"  Fetched records {start_index}–{start_index + len(gdf)}")

        if len(gdf) < page_size:
            break  # last page

        start_index += page_size

    combined = pd.concat(all_gdfs, ignore_index=True)
    print(f"Total gemeenten: {len(combined)}")
    return combined


def get_gemeente_by_code(gemeente_code):
    """Fetch gemeente boundary by code using client-side filtering"""
    print(f"Loading all gemeenten...")
    all_gemeenten = get_all_gemeenten()

    result = all_gemeenten[all_gemeenten["statcode"] == gemeente_code].copy()

    if result.empty:
        print(f"Code {gemeente_code} not found")
        print("Sample codes:", all_gemeenten["statcode"].head(10).tolist())
        return None

    print(f"Found:  {result['statnaam'].values[0]}")
    print(f"Code:   {result['statcode'].values[0]}")
    print(f"Bounds: {result.total_bounds}")
    return result


utrecht = get_gemeente_by_code("GM0344")

Loading all gemeenten...
  Fetched records 0–100
  Fetched records 100–200
  Fetched records 200–300
  Fetched records 300–342
Total gemeenten: 342
Found:  Utrecht
Code:   GM0344
Bounds: [126434.772  448709.995  141834.432  461609.5948]


In [6]:
import requests
import geopandas as gpd
import pandas as pd
from io import BytesIO

# Step 1: get Utrecht boundary (using your working function)
utrecht = get_gemeente_by_code("GM0344")

# Step 2: extract bounding box in RD New coordinates
minx, miny, maxx, maxy = utrecht.total_bounds
bbox_str = f"{minx:.0f},{miny:.0f},{maxx:.0f},{maxy:.0f}"
print(f"Bounding box: {bbox_str}")

# Step 3: query boreholes within bbox
def get_boreholes_in_bbox(bbox_str, page_size=100):
    """Fetch all boreholes within a bounding box with pagination"""
    url = "https://service.pdok.nl/bzk/bro-bodemkundigboringprofiel/wfs/v1_0"
    all_gdfs = []
    start_index = 0

    while True:
        params = {
            "service":      "WFS",
            "version":      "2.0.0",
            "request":      "GetFeature",
            "typeName":     "bro-bodemkundigboringprofiel:bodemkundigboringprofiel",
            "bbox":         bbox_str,
            "outputFormat": "application/json",
            "count":        page_size,
            "startIndex":   start_index
        }

        response = requests.get(url, params=params)
        print(f"  Page {start_index // page_size + 1}: status {response.status_code}")

        if response.status_code != 200:
            print(f"  Error: {response.text[:200]}")
            break

        gdf = gpd.read_file(BytesIO(response.content))

        if gdf.empty:
            break

        all_gdfs.append(gdf)
        print(f"  Fetched {len(gdf)} boreholes (total so far: {sum(len(g) for g in all_gdfs)})")

        if len(gdf) < page_size:
            break

        start_index += page_size

    if not all_gdfs:
        return None

    combined = pd.concat(all_gdfs, ignore_index=True)
    return combined

boreholes_bbox = get_boreholes_in_bbox(bbox_str)

# Step 4: clip to exact Utrecht boundary
if boreholes_bbox is not None:
    boreholes_bbox = boreholes_bbox.to_crs(utrecht.crs)
    boreholes = gpd.clip(boreholes_bbox, utrecht)
    print(f"\nBoreholes in bbox:    {len(boreholes_bbox)}")
    print(f"Boreholes in Utrecht: {len(boreholes)}")
    print(f"\nColumns: {boreholes.columns.tolist()}")
    print(boreholes.head())

Loading all gemeenten...
  Fetched records 0–100
  Fetched records 100–200
  Fetched records 200–300
  Fetched records 300–342
Total gemeenten: 342
Found:  Utrecht
Code:   GM0344
Bounds: [126434.772  448709.995  141834.432  461609.5948]
Bounding box: 126435,448710,141834,461610
  Page 1: status 404
  Error: 404 page not found



In [7]:
import requests

# Search PDOK for the correct BRO borehole service
url = "https://api.pdok.nl/datasets"

response = requests.get(url)
print(response.status_code)
print(response.text[:3000])

404
<html>
<head><title>404 Not Found</title></head>
<body>
<center><h1>404 Not Found</h1></center>
<hr><center>nginx/1.27.0</center>
</body>
</html>



In [1]:
## attempting to import REGIS data programatically
import xarray as xr
import numpy as np

ds = xr.open_dataset("https://dinodata.nl/opendap/REGIS/REGIS.nc")

# Your location in RD coordinates
x_target, y_target = 142862, 432871

# Find nearest grid point
x_idx = np.argmin(np.abs(ds.x.values - x_target))
y_idx = np.argmin(np.abs(ds.y.values - y_target))

# Print all layers at this point
for layer in ds.layer.values:
    top = ds['top'].sel(layer=layer).values[y_idx, x_idx]
    bot = ds['bottom'].sel(layer=layer).values[y_idx, x_idx]
    kh  = ds['kh'].sel(layer=layer).values[y_idx, x_idx]
    if not np.isnan(top):
        print(f"{layer:20s}  top: {top:7.1f} m NAP  bot: {bot:7.1f} m NAP  kh: {kh:.1f} m/d")

ERR: curl error: SSL peer certificate or SSH remote key was not OK
curl error details: 


OSError: [Errno -68] NetCDF: I/O failure: 'https://dinodata.nl/opendap/REGIS/REGIS.nc'